# Tree Target Generation

Generate randomly placed tree target points next to a recorded trajectory and store them inside the trajectory object.

In [66]:
import numpy as np
import dill
from pathlib import Path
import plotly.graph_objects as go

trajectory_path = Path("gt_coverage.dill")  # Input trajectory relative to this notebook
random_seed = 42  # Change to regenerate a different set of target points

rng = np.random.default_rng(random_seed)


In [67]:
with open(trajectory_path, "rb") as f:
    trajectory = dill.load(f)

poses = np.array([pose[:3, 3] for pose in trajectory.poses_se3])
timestamps = trajectory.timestamps

print(f"Loaded {len(poses)} poses from {trajectory_path}")
if len(timestamps) > 1:
    print(f"Time span: {timestamps[0]:.2f}s -> {timestamps[-1]:.2f}s")


Loaded 2741 poses from gt_coverage.dill
Time span: 0.00s -> 274.11s


In [68]:
def direction_at_index(idx, pose_array):
    prev_idx = max(0, idx - 1)
    next_idx = min(len(pose_array) - 1, idx + 1)
    tangent = pose_array[next_idx] - pose_array[prev_idx]
    tangent[2] = 0.0
    norm = np.linalg.norm(tangent[:2])
    if norm < 1e-6:
        return None
    return tangent / norm


def generate_target_points(
    poses,
    rng,
    num_points=60,
    lateral_range=(2.0, 6.0),
    min_spacing=1.5,
    max_attempt_factor=30,
    cluster_probability=0.4,
    cluster_size_range=(3, 8),
    cluster_spread=0.8,
    cluster_spacing_factor=0.35,
):
    targets = []
    attempts = 0
    max_attempts = max_attempt_factor * max(1, num_points)
    cluster_min = max(1, int(min(cluster_size_range)))
    cluster_max = max(cluster_min, int(max(cluster_size_range)))
    cluster_spread = max(float(cluster_spread), 1e-3)
    inner_spacing = max(min_spacing * float(cluster_spacing_factor), 0.01)

    while len(targets) < num_points and attempts < max_attempts:
        attempts += 1
        idx = int(rng.integers(1, len(poses) - 1))
        base_point = poses[idx].copy()
        direction = direction_at_index(idx, poses)
        if direction is None:
            continue
        lateral = np.array([-direction[1], direction[0], 0.0])
        lateral_norm = np.linalg.norm(lateral[:2])
        if lateral_norm < 1e-6:
            continue
        lateral /= lateral_norm
        distance = rng.uniform(*lateral_range)
        side = rng.choice([-1.0, 1.0])
        base_candidate = base_point + side * distance * lateral
        base_candidate[2] = base_point[2]

        spawn_cluster = rng.random() < cluster_probability
        cluster_count = 1
        if spawn_cluster:
            cluster_count = int(rng.integers(cluster_min, cluster_max + 1))
        cluster_count = max(1, min(cluster_count, num_points - len(targets)))

        local_offsets = [np.zeros(2)]
        if cluster_count > 1:
            local_offsets.extend(rng.normal(scale=cluster_spread, size=(cluster_count - 1, 2)))

        for offset_id, offset in enumerate(local_offsets):
            candidate = base_candidate + offset[0] * direction + offset[1] * lateral
            candidate[2] = base_candidate[2]
            if targets:
                threshold = min_spacing if (offset_id == 0 or cluster_count == 1) else inner_spacing
                distances = np.linalg.norm(np.array(targets) - candidate, axis=1)
                if np.any(distances < threshold):
                    continue
            targets.append(candidate.copy())
            if len(targets) >= num_points:
                break

    if len(targets) < num_points:
        print(
            f"Warning: requested {num_points} targets but only generated {len(targets)}. Consider adjusting the parameters."
        )
    return np.array(targets)


In [69]:
num_target_points = 600
lateral_offset_range = (0, 10)
min_spacing = 0.1

# Cluster-Konfiguration
cluster_probability = 0.2  # 0 = keine Cluster, 1 = immer Cluster
cluster_size_range = (2, 20)  # minimale und maximale Punkte pro Cluster
cluster_spread = 5  # Standardabweichung der lokalen Versätze [m]
cluster_spacing_factor = 0.3  # Faktor für Mindestabstand innerhalb eines Clusters

target_points = generate_target_points(
    poses,
    rng,
    num_points=num_target_points,
    lateral_range=lateral_offset_range,
    min_spacing=min_spacing,
    cluster_probability=cluster_probability,
    cluster_size_range=cluster_size_range,
    cluster_spread=cluster_spread,
    cluster_spacing_factor=cluster_spacing_factor,
)

print(f"Generated {len(target_points)} target points")


Generated 600 target points


In [70]:
trajectory.meta["TargetPoints"] = target_points
trajectory.meta["TargetPointParameters"] = {
    "num_target_points": num_target_points,
    "lateral_offset_range": lateral_offset_range,
    "min_spacing": min_spacing,
    "cluster_probability": cluster_probability,
    "cluster_size_range": cluster_size_range,
    "cluster_spread": cluster_spread,
    "cluster_spacing_factor": cluster_spacing_factor,
}
print("Stored target points in trajectory.meta['TargetPoints']")
print(target_points[:5])


Stored target points in trajectory.meta['TargetPoints']
[[ 97.96100712 154.57606365  -0.25352954]
 [ 95.1568963  151.2295037   -0.43909847]
 [ 59.05919734  57.96472781  15.11383526]
 [ 62.99485043  62.31997316  15.11383526]
 [ 58.75846672  63.60258031  15.11383526]]


In [71]:
fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=poses[:, 0],
        y=poses[:, 1],
        z=poses[:, 2],
        mode="lines",
        name="Trajectory",
        line=dict(color="steelblue", width=4),
    )
)

if len(target_points):
    fig.add_trace(
        go.Scatter3d(
            x=target_points[:, 0],
            y=target_points[:, 1],
            z=target_points[:, 2],
            mode="markers",
            name="Target points",
            marker=dict(color="forestgreen", size=4),
        )
    )

fig.update_layout(
    title="Trajectory with randomly placed tree targets",
    scene=dict(
        xaxis_title="X [m]",
        yaxis_title="Y [m]",
        zaxis_title="Z [m]",
        aspectmode="data",
    ),
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()


In [72]:
output_path = trajectory_path.with_name(f"{trajectory_path.stem}withTargets.dill")

with open(output_path, "wb") as f:
    dill.dump(trajectory, f)

print(f"Trajectory with targets saved to {output_path}")


Trajectory with targets saved to gt_coveragewithTargets.dill
